# AquaPINN Sequim Quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cwq2016/AquaPINN_tmp/blob/main/Examples/Colab/01_Sequim_Quickstart.ipynb)

Run the cells from top to bottom. The notebook reads `Examples/Sequim/run/params.toml` as-is; it has no separate parameter form. To change the tag, time window, method, or file paths, edit that TOML file before running.


## 1. Set up

Run the setup cell once. It installs the Python dependencies, finds this checkout (or clones it in Colab), and mounts Google Drive so the result folder can be kept after the session ends.


In [ ]:
%pip -q install "numpy<2.5" scipy pandas matplotlib utm networkx scikit-learn tqdm six

from pathlib import Path
import subprocess, sys

# Reuse the current checkout, or clone the public repository in Colab.
REPO_URL = "https://github.com/cwq2016/AquaPINN_tmp.git"
REPO_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "AquaPINN_Tracker3D").is_dir()), None)
if REPO_DIR is None:
    REPO_DIR = Path("/content/AquaPINN_tmp")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                        REPO_URL, str(REPO_DIR)], check=True)

# Colab sessions are temporary, so use Drive for the saved result bundle.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUTPUT = Path("/content/drive/MyDrive/AquaPINN_Results")
except ImportError:
    DRIVE_OUTPUT = None
print("Project ready:", REPO_DIR)


## 2. Load parameters and run

The source launcher expects paths relative to `Examples/Sequim/run`, so this cell runs from that folder. It reads the settings directly from `params.toml`, converts the selected MAT detections, then calls the tracker. Sequim detections are already time-corrected; this workflow does not run a new synchronization stage.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, shutil, sys, uuid
import pandas as pd
# Run from the example folder so relative paths in params.toml resolve correctly.
SEQ_RUN = REPO_DIR / "Examples/Sequim/run"
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(SEQ_RUN))

import AquaPINN_Tracker3D as pinn
from AquaPINN_Tracker3D import config
from AquaPINN_Tracker3D.SharedLibs.functions import load_params
from convertDetections import convert_mat_to_synced_pickle, find_mat_tag_indices

os.chdir(SEQ_RUN)
PARAMS_FILE = SEQ_RUN / "params.toml"
# Use the first parameter combination from the supplied TOML configuration.
train_params = load_params(str(PARAMS_FILE))[0]
print("Loaded parameters from:", PARAMS_FILE)
print("TagID:", train_params["TagID"], "| Method:", train_params["trackMethod"],
      "| Output folder:", train_params["output_folder"])

OUTPUT_DIR = Path(train_params["output_folder"])
if not OUTPUT_DIR.is_absolute():
    OUTPUT_DIR = (SEQ_RUN / OUTPUT_DIR).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Initialize the tracker and create its history log in the configured output folder.
config.oneKey_configure(use_cuda=True, precision=64, seed=42)
tag_id = int(train_params["TagID"])
pinn.getLogger(logFile=str(OUTPUT_DIR / f"history_Tag{tag_id}.log"))
phone_file = Path(train_params["phoneFile"]).resolve()
phone_synced = OUTPUT_DIR / (phone_file.stem + "_synced.csv")
if not phone_synced.exists():
    shutil.copy2(phone_file, phone_synced)
# Sequim already has corrected times, so reuse the receiver locations without clock synchronization.
loc_sys = pinn.localizationSystem().deploy(str(PARAMS_FILE), doSync=False)

# Read the MAT file from dataFolder; the example defaults to Examples/Sequim/data.
data_folder = Path(train_params["dataFolder"] or (SEQ_RUN.parent / "data"))
if not data_folder.is_absolute():
    data_folder = (SEQ_RUN / data_folder).resolve()
MAT_FILE = sorted(data_folder.glob("*.mat"))[0]
tag_codes = list(loc_sys.tagInfo["code"])
mat_indices = find_mat_tag_indices(str(MAT_FILE), tag_codes)
if mat_indices != sorted(mat_indices):
    raise ValueError("The tag catalogue order must match the MAT file tag order.")
synced_pickle = loc_sys.getSyncedDecodesFile(
    train_params["tagFile"], "Tag", newFolder=train_params["output_folder"])
# Convert the selected detections into the synced-pickle format expected by tracker.run.
convert_mat_to_synced_pickle(
    mat_file=str(MAT_FILE), out_pickle=synced_pickle,
    badPhones=train_params["badPhones"], tagIndices=mat_indices)
# Estimate the track using the method selected in params.toml.
pinn.tracker.run(str(PARAMS_FILE), loc_sys)

tag_code = str(tag_codes[tag_id]).strip()
track_method = "NNplus" if train_params["trackMethod"] == "NN" and train_params.get("nSamples", 1) > 1 else train_params["trackMethod"]
track_name = f"track_Tag{tag_id}_{tag_code}_{track_method}.csv"
TRACK_CSV = OUTPUT_DIR / "Track" / "track" / track_name
track_rel = Path(os.path.relpath(TRACK_CSV, SEQ_RUN)).as_posix()
run_record = {"schema": 1, "run_id": uuid.uuid4().hex, "state": "completed",
              "method": track_method, "track_csv": track_rel,
              "track_sha256": hashlib.sha256(TRACK_CSV.read_bytes()).hexdigest()}
(SEQ_RUN / "run.json").write_text(json.dumps(run_record, indent=2), encoding="utf-8")

# Keep a desktop-style folder on Drive in Colab; local Jupyter uses the configured output folder.
if DRIVE_OUTPUT is not None:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    SAVED_RUN = DRIVE_OUTPUT / f"Sequim_{stamp}"
    SAVED_RUN.mkdir(parents=True, exist_ok=False)
    shutil.copytree(OUTPUT_DIR, SAVED_RUN / "results")
    drive_record = dict(run_record, track_csv="results/Track/track/" + track_name)
    (SAVED_RUN / "run.json").write_text(json.dumps(drive_record, indent=2), encoding="utf-8")
    shutil.copy2(PARAMS_FILE, SAVED_RUN / "params.toml")
    print("Drive copy saved:", SAVED_RUN)
else:
    SAVED_RUN = SEQ_RUN
    print("Results saved:", OUTPUT_DIR)


## 3. Review results

The track CSV is the table of estimated positions; the pickle sidecar and figures are saved next to it. In Colab, the Drive bundle also contains `params.toml` and `run.json`, which points to the track file.


In [ ]:
from IPython.display import display, Image

# Preview the track table and the figures created by the tracker.
track = pd.read_csv(TRACK_CSV)
display(track.head())
for figure in sorted(OUTPUT_DIR.rglob("*.jpg"))[:4]:
    display(Image(filename=str(figure)))
print("Saved run folder:", SAVED_RUN)
